In [ ]:
# @title
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score

from google.colab import drive
drive.mount('/content/drive')

from IPython.display import HTML, display, clear_output
def set_css():
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)


Mounted at /content/drive


In [ ]:
# @title
root_dir = "/content/drive/MyDrive/"
pred_dir = Path(f"{root_dir}/final_results")

results = []

valid_labels = ["Complied", "Violated", "Not Applicable"]

prompt_type = {
    "t":  "Task Description",
    "v":  "Classification only",
    "r":  "With Explanation",
}

prompt_description = {

    "t1":  "Inline Classification Instruction",
    "t2":  "Constrained-Output Instruction",
    "t3":  "t2 - Alt Wording",
    "t4":  "t2 - Analysis-focused",

    "v1":  "t4 - JSON-Dash List",
    "v1.1":  "t4 - JSON-Plain Text",
    "v1.2":  "t4 - JSON-Brackets",

    "v2":  "v1 + Explicit Conditional Instruction",
    "v2.0.1":  "v1 + Conditional Instruction",
    "v2.0.2":  "v1 + Direct Instruction",

    "v2.1":  "v1.1 + Explicit Conditional Instruction",
    "v2.1.1":  "v1.1 + Conditional Instruction",
    "v2.1.2":  "v1.1 + Direct Instruction",

    "v2.2":  "v1.2 + Explicit Conditional Instruction",
    "v2.2.1":  "v1.2 + Conditional Instruction",
    "v2.2.2":  "v1.2 + Direct Instruction",

    "v3":  "v2 + Ambiguity-Degree Instruction",
    "v3.0.1":  "v2 + Ambiguity-Free Instruction",
    "v3.0.2": "v2 + Confidence Instruction",

    "v3.1":  "v2.1 + Ambiguity-Degree Instruction",
    "v3.1.1":  "v2.1 + Ambiguity-Free Instruction",
    "v3.1.2": "v2.1 + Confidence Instruction",

    "v3.2":  "v2.2 + Ambiguity-Degree Instruction",
    "v3.2.1":  "v2.2 + Ambiguity-Free Instruction",
    "v3.2.2": "v2.2 + Confidence Instruction",

    "v4":  "v3 + Implicit COT Before Answering",
    "v4.0.1":  "v3 + Implicit COT-Image First",
    "v4.0.2":  "v3 + Implicit COT-Rule First",
    "v4.0.3":  "v3 + Implicit COT",

    "v4.1":  "v3.1 + Implicit COT Before Answering",
    "v4.1.1":  "v3.1 + Implicit COT-Image First",
    "v4.1.2":  "v3.1 + Implicit COT-Rule First",
    "v4.1.3":  "v3.1 + Implicit COT",

    "v4.2":  "v3.2 + Implicit COT Before Answering",
    "v4.2.1":  "v3.2 + Implicit COT-Image First",
    "v4.2.2":  "v3.2 + Implicit COT-Rule First",
    "v4.2.3":  "v3.2 + Implicit COT",

    # "r1.0.1":  "v3 + Explanation-One Sentence",

    "r1":  "v4 + Explanation",
    "r1.0.1":  "v4.0.1 + Explanation",
    "r1.0.2":  "v4.0.2 + Explanation",
    "r1.0.3":  "v4.0.3 + Explanation",
    "r1.0.4":  "v3 + Explanation",
    "r1.0.5":  "v1 + Explanation",


    # "v5":  "Short free-form reasoning",
    # "v6":  "Short guided reasoning",
    # "v7":  "Detailed stepwise guided reasoning",
    # "v8":  "Structured XML-tagged output format",
    # "v9":  "Multi-rule assessment (based on v4)",
    # "v10": "Multi-rule assessment (based on v6)",
    # "v11": "Multi-rule assessment (based on v7)",
    # "v12": "Multi-rule assessment (based on v8)",
}

experiment_type = {
    "v1":  "Base",
    "v4":  "Prompt Engineered",
    "r1":  "Prompt Engineered",
    "r1.0.5":  "Base"
}


for pred_path in sorted(pred_dir.glob("*.csv")):

        try:
            merged = pd.read_csv(pred_path)
        except Exception as e:
            print(f"Error reading {pred_path}: {e}")
            continue

    # for rule in merged_['Rule'].unique():
    #     merged = merged_[merged_['Rule'] == rule]

        merged['Pred Label'] = merged['Pred Label'].fillna('Unknown')
        valid_labels = ["Complied", "Violated", "Not Applicable"]
        coverage = (merged['Pred Label'].isin(valid_labels)).mean()

        merged_valid = merged[merged['Pred Label'].isin(valid_labels)]

        if len(merged_valid) > 0:
            accuracy = accuracy_score(merged['Label'], merged['Pred Label'])
            f1_macro = f1_score(merged_valid['Label'], merged_valid['Pred Label'], average="macro")

        else:
            f1_macro = 0.0
            class_report = "No valid predictions"
        parts = pred_path.stem.split('-')
        if len(parts) == 6:
            set, domain, model_name, instruction, rule_template = parts[0], parts[1], parts[2], parts[3], parts[4]
            time_part = parts[-1]
            try:
                mins, secs = map(int, time_part[:-1].split('m'))
                inference_time_sec = mins * 60 + secs
            except ValueError:
                mins = secs = inference_time_sec = None
        else:
            domain = model_name = instruction = None
            mins = secs = inference_time_sec = None

        # if len(instruction.split('_')) == 1:
        #     instruction += '_zeroshot'

        results.append({
            "File": pred_path.name,
            "Set": set,
            "Domain": domain,
            # "Rule": f"{rule} ({domain.title()})",
            "Experiment Type": experiment_type.get(parts[3]),
            "Rule Template": rule_template.replace('-', ' ').title(),
            "Output Type": prompt_type.get(parts[3][0]),
            "Model": model_name,
            "Template ID": instruction,
            "Description": prompt_description.get(parts[3]),
            "F1 Macro": round(f1_macro, 4),
            "Accuracy": round(accuracy, 4),

            "Coverage": round(coverage, 4),
            "Inference Time (sec)": inference_time_sec/len(merged) if inference_time_sec else None,
        })

        summary_df = pd.DataFrame(results)


In [ ]:
summary_df[(summary_df['Model'] == 'llava') & (summary_df['Set'] == 'test_df') & (summary_df['Experiment Type'] == 'Base')]

,File,Set,Domain,Experiment Type,Rule Template,Output Type,Model,Template ID,Description,Accuracy,F1 Macro,Coverage,Inference Time (sec)
5,test_df-construction-llava-r1.0.5-coded_rules-...,test_df,construction,Base,Coded_Rules,With Explanation,llava,r1.0.5,v1 + Explanation,0.520,0.4307,1.0,2.092
6,test_df-construction-llava-v1-coded_rules-4m48...,test_df,construction,Base,Coded_Rules,Classification only,llava,v1,t4 - JSON-Dash List,0.398,0.3605,1.0,0.576
21,test_df-traffic-llava-r1.0.5-coded_rules-16m32...,test_df,traffic,Base,Coded_Rules,With Explanation,llava,r1.0.5,v1 + Explanation,0.660,0.6182,1.0,1.984
22,test_df-traffic-llava-v1-coded_rules-5m5s.csv,test_df,traffic,Base,Coded_Rules,Classification only,llava,v1,t4 - JSON-Dash List,0.604,0.5773,1.0,0.610
37,test_df-warehouse-llava-r1.0.5-coded_rules-17m...,test_df,warehouse,Base,Coded_Rules,With Explanation,llava,r1.0.5,v1 + Explanation,0.506,0.4411,1.0,2.138
38,test_df-warehouse-llava-v1-coded_rules-4m34s.csv,test_df,warehouse,Base,Coded_Rules,Classification only,llava,v1,t4 - JSON-Dash List,0.390,0.3421,1.0,0.548


Val

In [ ]:
agg_df = (
    summary_df[(summary_df['Set'] == 'val_df')].groupby(['Template ID', 'Output Type', 'Description'], as_index=False)
    # summary_df[(summary_df['Set'] == 'val_df')].groupby(['Domain', 'Template ID'], as_index=False)
    # summary_df[(summary_df['Set'] == 'val_df')].groupby(['Rule', 'Template ID'], as_index=False)
    # summary_df[(summary_df['Set'] == 'test_df')].groupby(['Model', 'Type', 'Description', 'Template ID'], as_index=False)

      .agg({
          # 'Model': 'nunique',
          # 'Domain': 'nunique',
          # 'Rule': 'nunique',
          'Accuracy': 'mean',
          'F1 Macro': 'mean',
          'Coverage': 'mean',
          'Inference Time (sec)': 'mean'
      })
      .rename(columns={
          'Accuracy': 'Avg Accuracy',
          'F1 Macro': 'Avg F1 Macro',
          'Inference Time (sec)': 'Avg Inference Time (sec)'
      }).round(4)
)

agg_df #.sort_values(by='Avg F1 Macro', ascending=False)

,Template ID,Output Type,Description,Avg Accuracy,Avg F1 Macro,Coverage,Avg Inference Time (sec)
0,r1,With Explanation,v4 + Explanation,0.6507,0.5655,1.0000,2.4987
1,r1.0.1,With Explanation,v4.0.1 + Explanation,0.6147,0.5562,1.0000,2.8247
2,r1.0.2,With Explanation,v4.0.2 + Explanation,0.6033,0.5506,1.0000,2.8940
3,r1.0.3,With Explanation,v4.0.3 + Explanation,0.6327,0.5485,1.0000,2.4580
4,r1.0.4,With Explanation,v3 + Explanation,0.6400,0.5569,1.0000,2.3493
5,r1.0.5,With Explanation,v1 + Explanation,0.5280,0.4716,1.0000,2.1107
6,t1,Task Description,Inline Classification Instruction,0.3067,0.2058,1.0000,0.1787
7,t2,Task Description,Constrained-Output Instruction,0.3913,0.2790,1.0000,0.1853
8,t3,Task Description,t2 - Alt Wording,0.4647,0.3402,1.0000,0.1920
9,t4,Task Description,t2 - Analysis-focused,0.4980,0.3794,1.0000,0.1953


Test

In [ ]:
summary_df[(summary_df['Set'] == 'test_df') & (summary_df['Template ID'] == 'v1') & (summary_df['Model'] == 'llava')]

,File,Set,Domain,Experiment Type,Rule Template,Output Type,Model,Template ID,Description,F1 Macro,Accuracy,Coverage,Inference Time (sec)
6,test_df-construction-llava-v1-coded_rules-4m48...,test_df,construction,Base,Coded_Rules,Classification only,llava,v1,t4 - JSON-Dash List,0.3605,0.398,1.0,0.576
22,test_df-traffic-llava-v1-coded_rules-5m5s.csv,test_df,traffic,Base,Coded_Rules,Classification only,llava,v1,t4 - JSON-Dash List,0.5773,0.604,1.0,0.610
38,test_df-warehouse-llava-v1-coded_rules-4m34s.csv,test_df,warehouse,Base,Coded_Rules,Classification only,llava,v1,t4 - JSON-Dash List,0.3421,0.390,1.0,0.548


In [ ]:
agg_df = (
    summary_df[(summary_df['Set'] == 'test_df')].groupby(['Model', 'Experiment Type', 'Template ID', 'Output Type', 'Description'], as_index=False)

      .agg({
          # 'Model': 'nunique',
          # 'Domain': 'nunique',
          # 'Rule': 'nunique',
          'Accuracy': 'mean',
          'F1 Macro': 'mean',
          'Coverage': 'mean',
          'Inference Time (sec)': 'mean'
      })
      .rename(columns={
          'Accuracy': 'Avg Accuracy',
          'F1 Macro': 'Avg F1 Macro',
          'Inference Time (sec)': 'Avg Inference Time (sec)'
      })
)

agg_df.sort_values(by=['Model', 'Template ID'], ascending=False) #.sort_values(by='Avg F1 Macro', ascending=False)

,Model,Experiment Type,Template ID,Output Type,Description,Avg Accuracy,Avg F1 Macro,Coverage,Avg Inference Time (sec)
15,llavanext,Prompt Engineered,v4,Classification only,v3 + Implicit COT Before Answering,0.650667,0.421100,1.000000,1.347333
13,llavanext,Base,v1,Classification only,t4 - JSON-Dash List,0.625333,0.494700,0.963333,0.947333
12,llavanext,Base,r1.0.5,With Explanation,v1 + Explanation,0.656667,0.555100,1.000000,3.590000
14,llavanext,Prompt Engineered,r1,With Explanation,v4 + Explanation,0.651333,0.421100,1.000000,3.851333
11,llavacot,Prompt Engineered,v4,Classification only,v3 + Implicit COT Before Answering,0.682667,0.633167,1.000000,10.246667
9,llavacot,Base,v1,Classification only,t4 - JSON-Dash List,0.552000,0.536633,1.000000,7.748667
8,llavacot,Base,r1.0.5,With Explanation,v1 + Explanation,0.607333,0.584633,1.000000,11.146000
10,llavacot,Prompt Engineered,r1,With Explanation,v4 + Explanation,0.679333,0.635833,1.000000,12.105333
7,llava,Prompt Engineered,v4,Classification only,v3 + Implicit COT Before Answering,0.659333,0.547900,1.000000,0.964667
5,llava,Base,v1,Classification only,t4 - JSON-Dash List,0.464000,0.426633,1.000000,0.578000


In [ ]:
agg_df[agg_df['Model'] == 'llava'].sort_values(by='Avg F1 Macro', ascending=False)

,Model,Experiment Type,Template ID,Output Type,Description,Avg Accuracy,Avg F1 Macro,Coverage,Avg Inference Time (sec)
6,llava,Prompt Engineered,r1,With Explanation,v4 + Explanation,0.656667,0.573833,1.0,2.356667
7,llava,Prompt Engineered,v4,Classification only,v3 + Implicit COT Before Answering,0.659333,0.547900,1.0,0.964667
4,llava,Base,r1.0.5,With Explanation,v1 + Explanation,0.562000,0.496667,1.0,2.071333
5,llava,Base,v1,Classification only,t4 - JSON-Dash List,0.464000,0.426633,1.0,0.578000


In [ ]:
agg_df[agg_df['Model'] == 'llavanext'].sort_values(by='Avg F1 Macro', ascending=False)

,Model,Experiment Type,Template ID,Output Type,Description,Avg Accuracy,Avg F1 Macro,Coverage,Avg Inference Time (sec)
4,llavanext,Base,r1.0.5,With Explanation,v1 + Explanation,0.656667,0.5551,1.000000,3.590000
5,llavanext,Base,v1,Classification only,t4 - JSON-Dash List,0.625333,0.4947,0.963333,0.947333
6,llavanext,Prompt Engineered,r1,With Explanation,v4 + Explanation,0.651333,0.4211,1.000000,3.851333
7,llavanext,Prompt Engineered,v4,Classification only,v3 + Implicit COT Before Answering,0.650667,0.4211,1.000000,1.347333


In [ ]:
agg_df[agg_df['Model'] == 'llamavision'].sort_values(by='Avg F1 Macro', ascending=False)

,Model,Experiment Type,Template ID,Output Type,Description,Avg Accuracy,Avg F1 Macro,Coverage,Avg Inference Time (sec)
2,llamavision,Prompt Engineered,r1,With Explanation,v4 + Explanation,0.362667,0.434067,0.823333,5.892667
3,llamavision,Prompt Engineered,v4,Classification only,v3 + Implicit COT Before Answering,0.342667,0.433167,0.794667,3.249333
0,llamavision,Base,r1.0.5,With Explanation,v1 + Explanation,0.330000,0.344467,0.931333,3.629333
1,llamavision,Base,v1,Classification only,t4 - JSON-Dash List,0.326667,0.316100,0.988000,0.707333


In [ ]:
agg_df[agg_df['Model'] == 'llavacot'].sort_values(by='Avg F1 Macro', ascending=False)

,Model,Experiment Type,Template ID,Output Type,Description,Avg Accuracy,Avg F1 Macro,Coverage,Avg Inference Time (sec)
10,llavacot,Prompt Engineered,r1,With Explanation,v4 + Explanation,0.679333,0.635833,1.0,12.105333
11,llavacot,Prompt Engineered,v4,Classification only,v3 + Implicit COT Before Answering,0.682667,0.633167,1.0,10.246667
8,llavacot,Base,r1.0.5,With Explanation,v1 + Explanation,0.607333,0.584633,1.0,11.146000
9,llavacot,Base,v1,Classification only,t4 - JSON-Dash List,0.552000,0.536633,1.0,7.748667


In [ ]:
# # summary_df[(agg_df['Template ID'].apply(lambda x:len(x)) == 2)].sort_values(by='Avg F1 Macro', ascending=False)
# summary_df[(summary_df['Template ID'].apply(lambda x:len(x)) == 2) & (summary_df['Set'] == 'test_df')].loc[summary_df[(summary_df['Template ID'].apply(lambda x:len(x)) == 2) & (summary_df['Set'] == 'test_df')].groupby('Domain')['F1 Macro'].idxmax()].reset_index(drop=True).sort_values(by='F1 Macro', ascending=False)[['Domain', 'Template ID', 'Accuracy', 'F1 Macro', 'Coverage', 'Inference Time (sec)']]
# # summary_df[(summary_df['Template ID'].apply(lambda x:len(x)) == 2) & (summary_df['Set'] == 'val_df')].loc[summary_df[(summary_df['Template ID'].apply(lambda x:len(x)) == 2) & (summary_df['Set'] == 'val_df')].groupby('Rule')['F1 Macro'].idxmax()].reset_index(drop=True).sort_values(by='F1 Macro', ascending=False)[['Rule', 'Template ID', 'Accuracy', 'F1 Macro', 'Coverage', 'Inference Time (sec)']]


Rule-Domain

In [ ]:
# @title
root_dir = "/content/drive/MyDrive/"
pred_dir = Path(f"{root_dir}/final_results")

results = []

valid_labels = ["Complied", "Violated", "Not Applicable"]

prompt_type = {
    "t":  "Task Description",
    "v":  "Classification only",
    "r":  "With Explanation",
}

prompt_description = {

    "t1":  "Inline Classification Instruction",
    "t2":  "Constrained-Output Instruction",
    "t3":  "t2 - Alt Wording",
    "t4":  "t2 - Analysis-focused",

    "v1":  "t4 - JSON-Dash List",
    "v1.1":  "t4 - JSON-Plain Text",
    "v1.2":  "t4 - JSON-Brackets",

    "v2":  "v1 + Explicit Conditional Instruction",
    "v2.0.1":  "v1 + Conditional Instruction",
    "v2.0.2":  "v1 + Direct Instruction",

    "v2.1":  "v1.1 + Explicit Conditional Instruction",
    "v2.1.1":  "v1.1 + Conditional Instruction",
    "v2.1.2":  "v1.1 + Direct Instruction",

    "v2.2":  "v1.2 + Explicit Conditional Instruction",
    "v2.2.1":  "v1.2 + Conditional Instruction",
    "v2.2.2":  "v1.2 + Direct Instruction",

    "v3":  "v2 + Ambiguity-Degree Instruction",
    "v3.0.1":  "v2 + Ambiguity-Free Instruction",
    "v3.0.2": "v2 + Confidence Instruction",

    "v3.1":  "v2.1 + Ambiguity-Degree Instruction",
    "v3.1.1":  "v2.1 + Ambiguity-Free Instruction",
    "v3.1.2": "v2.1 + Confidence Instruction",

    "v3.2":  "v2.2 + Ambiguity-Degree Instruction",
    "v3.2.1":  "v2.2 + Ambiguity-Free Instruction",
    "v3.2.2": "v2.2 + Confidence Instruction",

    "v4":  "v3 + Implicit COT Before Answering",
    "v4.0.1":  "v3 + Implicit COT-Image First",
    "v4.0.2":  "v3 + Implicit COT-Rule First",
    "v4.0.3":  "v3 + Implicit COT",

    "v4.1":  "v3.1 + Implicit COT Before Answering",
    "v4.1.1":  "v3.1 + Implicit COT-Image First",
    "v4.1.2":  "v3.1 + Implicit COT-Rule First",
    "v4.1.3":  "v3.1 + Implicit COT",

    "v4.2":  "v3.2 + Implicit COT Before Answering",
    "v4.2.1":  "v3.2 + Implicit COT-Image First",
    "v4.2.2":  "v3.2 + Implicit COT-Rule First",
    "v4.2.3":  "v3.2 + Implicit COT",

    # "r1.0.1":  "v3 + Explanation-One Sentence",

    "r1":  "v4 + Explanation",
    "r1.0.1":  "v4.0.1 + Explanation",
    "r1.0.2":  "v4.0.2 + Explanation",
    "r1.0.3":  "v4.0.3 + Explanation",
    "r1.0.4":  "v3 + Explanation",
    "r1.0.5":  "v1 + Explanation",


    # "v5":  "Short free-form reasoning",
    # "v6":  "Short guided reasoning",
    # "v7":  "Detailed stepwise guided reasoning",
    # "v8":  "Structured XML-tagged output format",
    # "v9":  "Multi-rule assessment (based on v4)",
    # "v10": "Multi-rule assessment (based on v6)",
    # "v11": "Multi-rule assessment (based on v7)",
    # "v12": "Multi-rule assessment (based on v8)",
}

experiment_type = {
    "v1":  "Base",
    "v4":  "Prompt Engineered",
    "r1":  "Prompt Engineered",
    "r1.0.5":  "Base"
}


for pred_path in sorted(pred_dir.glob("*.csv")):

    try:
        merged_ = pd.read_csv(pred_path)
    except Exception as e:
        print(f"Error reading {pred_path}: {e}")
        continue

    for rule in merged_['Rule'].unique():
        merged = merged_[merged_['Rule'] == rule]


        valid_labels = ["Complied", "Violated", "Not Applicable"]
        coverage = (merged['Pred Label'].isin(valid_labels)).mean()

        merged_valid = merged[merged['Pred Label'].isin(valid_labels)]

        if len(merged_valid) > 0:
            accuracy = accuracy_score(merged['Label'], merged['Pred Label'])
            f1_macro = f1_score(merged_valid['Label'], merged_valid['Pred Label'], average="macro")

        else:
            f1_macro = 0.0
            class_report = "No valid predictions"
        parts = pred_path.stem.split('-')
        if len(parts) == 6:
            set, domain, model_name, instruction, rule_template = parts[0], parts[1], parts[2], parts[3], parts[4]
            time_part = parts[-1]
            try:
                mins, secs = map(int, time_part[:-1].split('m'))
                inference_time_sec = mins * 60 + secs
            except ValueError:
                mins = secs = inference_time_sec = None
        else:
            domain = model_name = instruction = None
            mins = secs = inference_time_sec = None

        # if len(instruction.split('_')) == 1:
        #     instruction += '_zeroshot'

        results.append({
            "File": pred_path.name,
            "Set": set,
            "Domain": domain,
            "Rule": f"{rule} ({domain.title()})",
            "Experiment Type": experiment_type.get(parts[3]),
            "Rule Template": rule_template.replace('-', ' ').title(),
            "Output Type": prompt_type.get(parts[3][0]),
            "Model": model_name,
            "Template ID": instruction,
            "Description": prompt_description.get(parts[3]),
            "Accuracy": round(accuracy, 4),
            "F1 Macro": round(f1_macro, 4),
            "Coverage": round(coverage, 4),
            "Inference Time (sec)": inference_time_sec/len(merged) if inference_time_sec else None,
        })

        summary_df = pd.DataFrame(results)

# summary_df[(summary_df['Template ID'] == 'r1') & (summary_df['Set'] == 'test_df')].sort_values(by='F1 Macro', ascending=False)[['Rule', 'Template ID', 'Accuracy', 'F1 Macro', 'Coverage', 'Inference Time (sec)']]

In [ ]:
agg_df = (
    # summary_df[(summary_df['Set'] == 'val_df')].groupby(['Domain', 'Model', 'Template ID'], as_index=False)
    summary_df[(summary_df['Set'] == 'val_df')].groupby(['Rule', 'Model', 'Template ID'], as_index=False)

      .agg({
          # 'Model': 'nunique',
          # 'Domain': 'nunique',
          # 'Rule': 'nunique',
          'Accuracy': 'mean',
          'F1 Macro': 'mean',
          'Coverage': 'mean',
          'Inference Time (sec)': 'mean'
      })
      .rename(columns={
          'Accuracy': 'Avg Accuracy',
          'F1 Macro': 'Avg F1 Macro',
          'Inference Time (sec)': 'Avg Inference Time (sec)'
      })
)

agg_df[(agg_df['Template ID']=='r1') & (agg_df['Model'] == 'llava')].sort_values(by='Avg F1 Macro', ascending=False)

,Rule,Model,Template ID,Avg Accuracy,Avg F1 Macro,Coverage,Avg Inference Time (sec)
43,Driving Distraction (Traffic),llava,r1,0.91,0.8183,1.0,13.27
258,Ladder Use (Warehouse),llava,r1,0.81,0.6801,1.0,12.38
516,Surface Condition (Warehouse),llava,r1,0.71,0.6735,1.0,12.38
172,Forklift Use (Warehouse),llava,r1,0.70,0.6181,1.0,12.38
602,Vehicle Load (Traffic),llava,r1,0.83,0.5962,1.0,13.27
215,Ladder Use (Construction),llava,r1,0.68,0.5853,1.0,11.83
129,Fire Risk (Construction),llava,r1,0.85,0.5595,1.0,11.83
301,Pedestrian Crossing (Traffic),llava,r1,0.80,0.4988,1.0,13.27
473,Scaffolding Risk (Construction),llava,r1,0.64,0.4794,1.0,11.83
430,Road Condition (Traffic),llava,r1,0.50,0.4656,1.0,13.27


In [ ]:
# agg_df.loc[agg_df.groupby('Rule')['Avg F1 Macro'].idxmax()].reset_index(drop=True).sort_values(by='Avg F1 Macro', ascending=False)
# agg_df.loc[agg_df.groupby('Domain')['Avg F1 Macro'].idxmax()].reset_index(drop=True).sort_values(by='Avg F1 Macro', ascending=False)

Finetune

In [ ]:
# @title
root_dir = "/content/drive/MyDrive/"
pred_dir = Path(f"{root_dir}/final_results_finetune")

results = []

valid_labels = ["Complied", "Violated", "Not Applicable"]

prompt_type = {
    "t":  "Task Description",
    "v":  "Classification only",
    "r":  "With Explanation",
}

prompt_description = {

    "t1":  "Inline Classification Instruction",
    "t2":  "Constrained-Output Instruction",
    "t3":  "t2 - Alt Wording",
    "t4":  "t2 - Analysis-focused",

    "v1":  "t4 - JSON-Dash List",
    "v1.1":  "t4 - JSON-Plain Text",
    "v1.2":  "t4 - JSON-Brackets",

    "v2":  "v1 + Explicit Conditional Instruction",
    "v2.0.1":  "v1 + Conditional Instruction",
    "v2.0.2":  "v1 + Direct Classification Instruction",

    "v2.1":  "v1.1 + Explicit Conditional Instruction",
    "v2.1.1":  "v1.1 + Conditional Instruction",
    "v2.1.2":  "v1.1 + Direct Classification Instruction",

    "v2.2":  "v1.2 + Explicit Conditional Instruction",
    "v2.2.1":  "v1.2 + Conditional Instruction",
    "v2.2.2":  "v1.2 + Direct Classification Instruction",

    "v3":  "v2 + Ambiguity-Degree Instruction",
    "v3.0.1":  "v2 + Ambiguity-Free Instruction",
    "v3.0.2": "v2 + Confidence Instruction",

    "v3.1":  "v2.1 + Ambiguity-Degree Instruction",
    "v3.1.1":  "v2.1 + Ambiguity-Free Instruction",
    "v3.1.2": "v2.1 + Confidence Instruction",

    "v3.2":  "v2.2 + Ambiguity-Degree Instruction",
    "v3.2.1":  "v2.2 + Ambiguity-Free Instruction",
    "v3.2.2": "v2.2 + Confidence Instruction",

    "v4":  "v3 + Implicit COT Before Answering",
    "v4.0.1":  "v3 + Implicit COT-Image First",
    "v4.0.2":  "v3 + Implicit COT-Rule First",
    "v4.0.3":  "v3 + Implicit COT",

    "v4.1":  "v3.1 + Implicit COT Before Answering",
    "v4.1.1":  "v3.1 + Implicit COT-Image First",
    "v4.1.2":  "v3.1 + Implicit COT-Rule First",
    "v4.1.3":  "v3.1 + Implicit COT",

    "v4.2":  "v3.2 + Implicit COT Before Answering",
    "v4.2.1":  "v3.2 + Implicit COT-Image First",
    "v4.2.2":  "v3.2 + Implicit COT-Rule First",
    "v4.2.3":  "v3.2 + Implicit COT",

    # "r1.0.1":  "v3 + Explanation-One Sentence",

    "r1":  "v4 + Explanation",
    "r1.0.1":  "v4.0.1 + Explanation",
    "r1.0.2":  "v4.0.2 + Explanation",
    "r1.0.3":  "v4.0.3 + Explanation",
    "r1.0.4":  "v3 + Explanation",
    "r1.0.5":  "v1 + Explanation",


    # "v5":  "Short free-form reasoning",
    # "v6":  "Short guided reasoning",
    # "v7":  "Detailed stepwise guided reasoning",
    # "v8":  "Structured XML-tagged output format",
    # "v9":  "Multi-rule assessment (based on v4)",
    # "v10": "Multi-rule assessment (based on v6)",
    # "v11": "Multi-rule assessment (based on v7)",
    # "v12": "Multi-rule assessment (based on v8)",
}

prompt_description = {
    "v1": "Base",
    "v4": "Prompt Engineered",
    "r1": "Prompt Engineered",
    "r1.0.5": "Base"
}

for pred_path in sorted(pred_dir.glob("*.csv")):

        try:
            merged = pd.read_csv(pred_path)
        except Exception as e:
            print(f"Error reading {pred_path}: {e}")
            continue

    # for rule in merged_['Rule'].unique():
    #     merged = merged_[merged_['Rule'] == rule]


        valid_labels = ["Complied", "Violated", "Not Applicable"]
        coverage = (merged['Pred Label'].isin(valid_labels)).mean()

        merged_valid = merged[merged['Pred Label'].isin(valid_labels)]

        if len(merged_valid) > 0:
            accuracy = accuracy_score(merged['Label'], merged['Pred Label'])
            f1_macro = f1_score(merged_valid['Label'], merged_valid['Pred Label'], average="macro")

        else:
            f1_macro = 0.0
            class_report = "No valid predictions"
        parts = pred_path.stem.replace('coded-rules', 'coded_rules').split('-')
        if len(parts) == 6:
            set, domain, model_name, instruction, rule_template = parts[0], parts[1], parts[2], parts[3], parts[4]
            time_part = parts[-1]
            try:
                mins, secs = map(int, time_part[:-1].split('m'))
                inference_time_sec = mins * 60 + secs
            except ValueError:
                mins = secs = inference_time_sec = None
        else:
            domain = model_name = instruction = None
            mins = secs = inference_time_sec = None

        # if len(instruction.split('_')) == 1:
        #     instruction += '_zeroshot'

        results.append({
            "File": pred_path.name,
            "Set": set,
            "Domain": domain,
            # "Rule": f"{rule} ({domain.title()})",
            "Rule Template": rule_template.replace('-', ' ').title(),
            "Type": prompt_type.get(parts[3][0]),
            "Model": model_name,
            "Template ID": instruction.split('_')[0],
            "Training Data": instruction.split('_')[2],
            "Description": prompt_description.get(parts[3].split('_')[0]),
            "Accuracy": round(accuracy, 4),
            "F1 Macro": round(f1_macro, 4),
            "Coverage": round(coverage, 4),
            "Inference Time (sec)": inference_time_sec/len(merged) if inference_time_sec else None,
        })

        summary_df = pd.DataFrame(results)

In [ ]:
summary_df[summary_df['Domain'] == 'traffic'][['Domain', 'Model', 'Template ID', 'Type', 'Training Data',  'F1 Macro', 'Accuracy', 'Coverage', 'Inference Time (sec)']].sort_values(by='F1 Macro', ascending=False)

,Domain,Model,Template ID,Type,Training Data,F1 Macro,Accuracy,Coverage,Inference Time (sec)
4,traffic,llava,r1.0.5,With Explanation,TrafficOnly,0.8428,0.904,1.0,5.470
6,traffic,llava,v1,Classification only,TrafficMore,0.8368,0.896,1.0,0.938
7,traffic,llava,v1,Classification only,TrafficOnly,0.8089,0.898,1.0,0.844
5,traffic,llava,v1,Classification only,MergedOnly,0.8086,0.880,1.0,0.852


In [ ]:
summary_df[summary_df['Domain'] == 'warehouse'][['Domain', 'Model', 'Template ID', 'Type', 'Training Data', 'F1 Macro', 'Accuracy', 'Coverage', 'Inference Time (sec)']].sort_values(by='F1 Macro', ascending=False)

,Domain,Model,Template ID,Type,Training Data,F1 Macro,Accuracy,Coverage,Inference Time (sec)
11,warehouse,llava,v1,Classification only,WarehouseOnly,0.8679,0.896,1.0,0.830
8,warehouse,llava,r1.0.5,With Explanation,WarehouseOnly,0.8402,0.870,1.0,5.242
9,warehouse,llava,v1,Classification only,MergedOnly,0.8359,0.870,1.0,0.844
10,warehouse,llava,v1,Classification only,WarehouseMore,0.7503,0.832,1.0,0.936


In [ ]:
summary_df[summary_df['Domain'] == 'construction'][['Domain', 'Model', 'Template ID', 'Type', 'Training Data', 'F1 Macro', 'Accuracy', 'Coverage', 'Inference Time (sec)']].sort_values(by='F1 Macro', ascending=False)

,Domain,Model,Template ID,Type,Training Data,F1 Macro,Accuracy,Coverage,Inference Time (sec)
2,construction,llava,v1,Classification only,ConstructionOnly,0.8479,0.906,1.0,0.862
1,construction,llava,v1,Classification only,ConstructionMore,0.8375,0.906,1.0,0.934
0,construction,llava,r1.0.5,With Explanation,ConstructionOnly,0.8077,0.886,1.0,5.018
3,construction,llava,v1,Classification only,MergedOnly,0.8019,0.886,1.0,0.842


Active Learning

In [ ]:
results = []

root_dir = "/content/drive/MyDrive/"
pred_dir = Path(f"{root_dir}/final_results_active_learning/")
for pred_path in sorted(pred_dir.glob("*.csv")):
    if 'test_df' in str(pred_path):
        merged = pd.read_csv(pred_path)

        merged['Pred Label'] = merged['Pred Label'].fillna('Unknown')
        valid_labels = ["Complied", "Violated", "Not Applicable"]
        coverage = (merged['Pred Label'].isin(valid_labels)).mean()

        merged_valid = merged[merged['Pred Label'].isin(valid_labels)]

        if len(merged_valid) > 0:
            accuracy = accuracy_score(merged['Label'], merged['Pred Label'])
            f1_macro = f1_score(merged_valid['Label'], merged_valid['Pred Label'], average="macro")

        else:
            f1_macro = 0.0
            class_report = "No valid predictions"
        parts = pred_path.stem.split('-')
        if len(parts) == 7:
            set, domain, model_name, instruction, rule_template = parts[0], parts[1], parts[2], parts[3], parts[4]
            time_part = parts[-1]
            try:
                mins, secs = map(int, time_part[:-1].split('m'))
                inference_time_sec = mins * 60 + secs
            except ValueError:
                mins = secs = inference_time_sec = None
        else:
            domain = model_name = instruction = None
            mins = secs = inference_time_sec = None

        # if len(instruction.split('_')) == 1:
        #     instruction += '_zeroshot'

        results.append({
            "File": pred_path.name,
            "Set": set,
            "Domain": domain,
            # "Rule": f"{rule} ({domain.title()})",
            "Experiment Type": experiment_type.get(parts[3]),
            "Rule Template": rule_template.replace('-', ' ').title(),
            "Output Type": prompt_type.get(parts[3][0]),
            "Model": model_name,
            "Template ID": instruction,
            "Description": prompt_description.get(parts[3]),
            "F1 Macro": round(f1_macro, 4),
            "Accuracy": round(accuracy, 4),
            "Coverage": round(coverage, 4),
            "Inference Time (sec)": inference_time_sec/len(merged) if inference_time_sec else None,
        })

summary_df = pd.DataFrame(results)

In [ ]:
summary_df

,File,Set,Domain,Experiment Type,Rule Template,Output Type,Model,Template ID,Description,F1 Macro,Accuracy,Coverage,Inference Time (sec)
0,test_df-construction-llava-v1_AL_Round0_finetu...,test_df,construction,None,Coded,Classification only,llava,v1_AL_Round0_finetuned_ConstructionOnly2_class...,None,0.7680,0.848,1.0,0.870
1,test_df-construction-llava-v1_AL_Round0_finetu...,test_df,construction,None,Coded,Classification only,llava,v1_AL_Round0_finetuned_ConstructionOnly_classi...,None,0.7347,0.830,1.0,0.854
2,test_df-construction-llava-v1_AL_Round1_finetu...,test_df,construction,None,Coded,Classification only,llava,v1_AL_Round1_finetuned_ConstructionOnly2_class...,None,0.7977,0.868,1.0,0.972
3,test_df-construction-llava-v1_AL_Round2_finetu...,test_df,construction,None,Coded,Classification only,llava,v1_AL_Round2_finetuned_ConstructionOnly2_class...,None,0.8160,0.884,1.0,0.944
4,test_df-construction-llava-v1_AL_Round3_finetu...,test_df,construction,None,Coded,Classification only,llava,v1_AL_Round3_finetuned_ConstructionOnly2_class...,None,0.8392,0.898,1.0,0.792
5,test_df-traffic-llava-v1_AL_Round0_finetuned_T...,test_df,traffic,None,Coded,Classification only,llava,v1_AL_Round0_finetuned_TrafficOnly_classification,None,0.7954,0.856,1.0,0.950
6,test_df-traffic-llava-v1_AL_Round1_finetuned_T...,test_df,traffic,None,Coded,Classification only,llava,v1_AL_Round1_finetuned_TrafficOnly_classification,None,0.8268,0.878,1.0,0.814
7,test_df-traffic-llava-v1_AL_Round2_finetuned_T...,test_df,traffic,None,Coded,Classification only,llava,v1_AL_Round2_finetuned_TrafficOnly_classification,None,0.8401,0.890,1.0,0.854
8,test_df-traffic-llava-v1_AL_Round3_finetuned_T...,test_df,traffic,None,Coded,Classification only,llava,v1_AL_Round3_finetuned_TrafficOnly_classification,None,0.8416,0.892,1.0,0.962
9,test_df-warehouse-llava-v1_AL_Round0_finetuned...,test_df,warehouse,None,Coded,Classification only,llava,v1_AL_Round0_finetuned_WarehouseOnly_classific...,None,0.7734,0.818,1.0,0.942
